# Isolation Forest as Outlier Detection

## selected data set = "2_em_denoised.csv"

In [ ]:
import pandas as pd
em_filtered = pd.read_csv("2_em_denoised.csv")
print(em_filtered.shape)
print(em_filtered.dtypes)
print(em_filtered.columns.tolist())

## Preparing the Test Data

In [ ]:
import pandas as pd

def categorize_slopes(df: pd.DataFrame) -> dict:
    """
    Categorize slope values from grouped sensor readings into three categories:
    - slp00: all slopes in each group exactly zero and both gases absent
    - slp+ve: all slopes strictly positive (> 0.0) with either methane, ethylene, or both gases present
    - slp-ve: all slopes strictly negative (< 0.0) with either methane, ethylene, or both gases present

    Sensor-2 readings are excluded due to suspected faulty data.
    """

    # Define sensor groups (Sensor-2 excluded)
    sensor_groups = {
        "Sensor-1": ["Sensor-1-1(m)", "Sensor-1-2(m)", "Sensor-1-3(m)", "Sensor-1-4(m)"],            "Sensor-1": ["Sensor-1-1(m)", "Sensor-1-2(m)", "Sensor-1-3(m)", "Sensor-1-4(m)"],
        "Sensor-2": ["Sensor-2-1(m)", "Sensor-2-2(m)", "Sensor-2-3(m)", "Sensor-2-4(m)"],

        "Sensor-3": ["Sensor-3-1(m)", "Sensor-3-2(m)", "Sensor-3-3(m)", "Sensor-3-4(m)"],
        "Sensor-4": ["Sensor-4-1(m)", "Sensor-4-2(m)", "Sensor-4-3(m)", "Sensor-4-4(m)"],
    }

    # Initialize conditions
    zero_cond = pd.Series(True, index=df.index)
    pos_cond = pd.Series(True, index=df.index)
    neg_cond = pd.Series(True, index=df.index)

    # Apply slope conditions for each group
    for sensors in sensor_groups.values():
        for sensor in sensors:
            zero_cond &= (df[sensor] == 0.0)
            pos_cond &= (df[sensor] > 0.0)
            neg_cond &= (df[sensor] < 0.0)

    # Gas concentration conditions
    zero_cond &= (df["MethConcStart(ppm)"] == 0.0) & (df["EthyConcStart(ppm)"] == 0.0)

    # Positive slope: either methane OR ethylene OR both gases present
    pos_cond &= ((df["MethConcStart(ppm)"] != 0.0) | (df["EthyConcStart(ppm)"] != 0.0))

    # Negative slope: either methane OR ethylene OR both gases present
    neg_cond &= ((df["MethConcStart(ppm)"] != 0.0) | (df["EthyConcStart(ppm)"] != 0.0))

    return {
        "slp00": df[zero_cond],
        "slp+ve": df[pos_cond],
        "slp-ve": df[neg_cond]
    }

# --- Main script ---

# Load dataset
em_filt = pd.read_csv("2_em_denoised.csv", header=0)

# Convert all columns to numeric (non-numeric → NaN)
em_filt_2 = em_filt.apply(pd.to_numeric, errors="coerce")

# Categorize slopes
categories = categorize_slopes(em_filt_2)

# Save results to CSV
categories["slp00"].to_csv("3_slp_00__em_denoised.csv", index=False, float_format="%.6f")
categories["slp+ve"].to_csv("4_slp_pos_em_denoised.csv", index=False, float_format="%.6f")
categories["slp-ve"].to_csv("5_slp_neg_em_denoised.csv", index=False, float_format="%.6f")

# Print summary
print("Categorization complete. Files saved:")
for key, df_cat in categories.items():
    print(f" - {key}: {len(df_cat)} rows")


## Preparing Training Data

In [ ]:
import pandas as pd

def categorize_signals(df: pd.DataFrame) -> dict:
    """
    Categorize rows in a DataFrame based on Methane and Ethylene thresholds.
    Returns a dictionary of filtered DataFrames.
    """
    return {
        "sig00": df[
            (df["MethConcStart(ppm)"] == 0.0) &
            (df["EthyConcStart(ppm)"] == 0.0)
        ],

        "sig0E": df[
            (df["MethConcStart(ppm)"] == 0.0) &
            (df["EthyConcStart(ppm)"] > 0.0)
        ],

        "sigM0": df[
            (df["MethConcStart(ppm)"] > 0.0) &
            (df["EthyConcStart(ppm)"] == 0.0)
        ],

        "sigEM": df[
            (df["MethConcStart(ppm)"] > 0.0) &
            (df["EthyConcStart(ppm)"] > 0.0)
        ],
    }

# --- Main script ---

# Load dataset, ensure numeric values only
em_select = pd.read_csv("2_em_denoised.csv", header=0)

# Convert all columns to numeric where possible (non-numeric → NaN)
em_select_1 = em_select.apply(pd.to_numeric, errors="coerce")

# Drop rows with NaN (optional, ensures clean numeric dataset)
em_select_2 = em_select_1.dropna()

# Categorize signals
categories = categorize_signals(em_select_2)

# Save each category to CSV
categories["sig00"].to_csv("6_sig00_2_em_denoised.csv", index=False, float_format="%.6f")
categories["sig0E"].to_csv("7_sig0E_2_em_denoised.csv", index=False, float_format="%.6f")
categories["sigM0"].to_csv("8_sigM0_2_em_denoised.csv", index=False, float_format="%.6f")
categories["sigEM"].to_csv("9_sigEM_2_em_denoised.csv", index=False, float_format="%.6f")

# Print summary counts
print("Categorization complete. Row counts per category:")
for key, df_cat in categories.items():
    print(f"{key}: {len(df_cat)} rows")


## Normal Data = when both Ethylene and Methane gas concentration equals Zero

In [ ]:
import pandas as pd
df = pd.read_csv("6_sig00_2_em_denoised.csv")
print(df.head())
print(df.shape)

## Train the Model with Normal Data by Isolation Forest Algorithm

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split

# -----------------------------
# Define sensor groups
sensor_groups = {
    1: ["Sensor-1-1(m)", "Sensor-1-2(m)", "Sensor-1-3(m)", "Sensor-1-4(m)"],
    2: ["Sensor-2-1(m)", "Sensor-2-2(m)", "Sensor-2-3(m)", "Sensor-2-4(m)"],
    3: ["Sensor-3-1(m)", "Sensor-3-2(m)", "Sensor-3-3(m)", "Sensor-3-4(m)"],
    4: ["Sensor-4-1(m)", "Sensor-4-2(m)", "Sensor-4-3(m)", "Sensor-4-4(m)"],
}

gas_cols = ["MethConcStart(ppm)", "EthyConcStart(ppm)"]

# -----------------------------
def select_columns(df: pd.DataFrame, keep_cols: list) -> pd.DataFrame:
    """Return a DataFrame containing only the specified columns."""
    return df[keep_cols].copy()

# -----------------------------
# Start timer
start_time = time.time()

# -----------------------------
# Select file
selected_file = "6_sig00_2_em_denoised.csv"   # Change this to whichever file you want
df = pd.read_csv(selected_file)

# -----------------------------
# Choose sensor groups (example: groups 1 and 3)
selected_groups = [1,2, 3, 4]

# Collect columns from chosen groups
selected_sensor_cols = []
for g in selected_groups:
    selected_sensor_cols.extend(sensor_groups[g])

# Final feature set: selected sensors + gas
feature_cols = selected_sensor_cols 

# -----------------------------
# Select data
x = select_columns(df, feature_cols)

# -----------------------------
# Train/test split (10% test)
X_train, X_test = train_test_split(x, test_size=0.1, random_state=42)
# -----------------------------
# Save train and test sets
X_train.to_csv(f"X_train_{selected_file}", index=False, float_format="%.6f")
X_test.to_csv(f"X_test_{selected_file}", index=False, float_format="%.6f")

# -----------------------------
# Initialize Isolation Forest
isof = IsolationForest(
    n_estimators=300,
    contamination="auto",
    max_samples= 12000,  # 10% of the dataset (13704-0.1*13704)
    random_state=42
)

# -----------------------------
# Fit model
#%timeit model = isof.fit(X_train)
model = isof.fit(X_train)  # Fit the model
# -----------------------------
# Predict anomalies on test set
y_pred = model.predict(X_test)

# Count anomalies and normals
n_anomalies = np.sum(y_pred == -1)
n_normals   = np.sum(y_pred == 1)

# Percentages
pct_anomalies = n_anomalies / len(y_pred) * 100
pct_normals   = n_normals / len(y_pred) * 100

# -----------------------------
# End timer
end_time = time.time()

# -----------------------------
# Print results
print("Execution time:", end_time - start_time, "seconds")
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print(f"Files saved: X_train_{selected_file}, X_test_{selected_file}")
print(f"Anomalies: {n_anomalies} ({pct_anomalies:.2f}%)")
print(f"Normals:   {n_normals} ({pct_normals:.2f}%)")

## Model Evaluation

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split

# -----------------------------
# Define sensor groups
sensor_groups = {
    1: ["Sensor-1-1(m)", "Sensor-1-2(m)", "Sensor-1-3(m)", "Sensor-1-4(m)"],
    2: ["Sensor-2-1(m)", "Sensor-2-2(m)", "Sensor-2-3(m)", "Sensor-2-4(m)"],
    3: ["Sensor-3-1(m)", "Sensor-3-2(m)", "Sensor-3-3(m)", "Sensor-3-4(m)"],
    4: ["Sensor-4-1(m)", "Sensor-4-2(m)", "Sensor-4-3(m)", "Sensor-4-4(m)"],
}

gas_cols = ["MethConcStart(ppm)", "EthyConcStart(ppm)"]

# -----------------------------
def select_columns(df: pd.DataFrame, keep_cols: list) -> pd.DataFrame:
    """Return a DataFrame containing only the specified columns."""
    return df[keep_cols].copy()

# -----------------------------
# Start timer
start_time = time.time()

# -----------------------------
# Select file
selected_file = "4_slp_pos_em_denoised.csv"   # Change this to whichever file you want
df = pd.read_csv(selected_file)

# -----------------------------
# Choose sensor groups (example: groups 1 and 3)
selected_groups = [1,2, 3, 4]

# Collect columns from chosen groups
selected_sensor_cols = []
for g in selected_groups:
    selected_sensor_cols.extend(sensor_groups[g])

# Final feature set: selected sensors + gas
feature_cols = selected_sensor_cols 

# -----------------------------
# Select test data
x = select_columns(df, feature_cols)
x.to_csv(f"X_test_{selected_file}_{selected_groups}", index=False, float_format="%.6f")
# -----------------------------
# Predict anomalies on test set
y_pred = model.predict(x)

# Count anomalies and normals
n_anomalies = np.sum(y_pred == -1)
n_normals   = np.sum(y_pred == 1)

# Percentages
pct_anomalies = n_anomalies / len(y_pred) * 100
pct_normals   = n_normals / len(y_pred) * 100

# -----------------------------
# End timer
end_time = time.time()

# -----------------------------
# Print results
print("Execution time:", end_time - start_time, "seconds")
print("Test shape:", x.shape)
print(f"Files saved: X_test_{selected_file}_{selected_groups}")
print(f"Anomalies: {n_anomalies} ({pct_anomalies:.2f}%)")
print(f"Normals:   {n_normals} ({pct_normals:.2f}%)")

##  Improve the Performance by droping Sensor Group 2

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split

# -----------------------------
# Define sensor groups
sensor_groups = {
    1: ["Sensor-1-1(m)", "Sensor-1-2(m)", "Sensor-1-3(m)", "Sensor-1-4(m)"],
    2: ["Sensor-2-1(m)", "Sensor-2-2(m)", "Sensor-2-3(m)", "Sensor-2-4(m)"],
    3: ["Sensor-3-1(m)", "Sensor-3-2(m)", "Sensor-3-3(m)", "Sensor-3-4(m)"],
    4: ["Sensor-4-1(m)", "Sensor-4-2(m)", "Sensor-4-3(m)", "Sensor-4-4(m)"],
}

gas_cols = ["MethConcStart(ppm)", "EthyConcStart(ppm)"]

# -----------------------------
def select_columns(df: pd.DataFrame, keep_cols: list) -> pd.DataFrame:
    """Return a DataFrame containing only the specified columns."""
    return df[keep_cols].copy()

# -----------------------------
# Start timer
start_time = time.time()

# -----------------------------
# Select file
selected_file = "6_sig00_2_em_denoised.csv"   # Change this to whichever file you want
df = pd.read_csv(selected_file)

# -----------------------------
# Choose sensor groups (example: groups 1 and 3)
selected_groups = [1, 3, 4]

# Collect columns from chosen groups
selected_sensor_cols = []
for g in selected_groups:
    selected_sensor_cols.extend(sensor_groups[g])

# Final feature set: selected sensors + gas
feature_cols = selected_sensor_cols 

# -----------------------------
# Select data
x = select_columns(df, feature_cols)

# -----------------------------
# Train/test split (10% test)
X_train, X_test = train_test_split(x, test_size=0.1, random_state=42)
# -----------------------------
# Save train and test sets
X_train.to_csv(f"X_train_{selected_file}", index=False, float_format="%.6f")
X_test.to_csv(f"X_test_{selected_file}", index=False, float_format="%.6f")

# -----------------------------
# Initialize Isolation Forest
isof = IsolationForest(
    n_estimators=300,
    contamination="auto",
    max_samples= 12000,  # 10% of the dataset (13704-0.1*13704)
    random_state=42
)

# -----------------------------
# Fit model
#%timeit model = isof.fit(X_train)
model = isof.fit(X_train)  # Fit the model
# -----------------------------
# Predict anomalies on test set
y_pred = model.predict(X_test)

# Count anomalies and normals
n_anomalies = np.sum(y_pred == -1)
n_normals   = np.sum(y_pred == 1)

# Percentages
pct_anomalies = n_anomalies / len(y_pred) * 100
pct_normals   = n_normals / len(y_pred) * 100

# -----------------------------
# End timer
end_time = time.time()

# -----------------------------
# Print results
print("Execution time:", end_time - start_time, "seconds")
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print(f"Files saved: X_train_{selected_file}, X_test_{selected_file}")
print(f"Anomalies: {n_anomalies} ({pct_anomalies:.2f}%)")
print(f"Normals:   {n_normals} ({pct_normals:.2f}%)")

## Model Evaluation

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split

# -----------------------------
# Define sensor groups
sensor_groups = {
    1: ["Sensor-1-1(m)", "Sensor-1-2(m)", "Sensor-1-3(m)", "Sensor-1-4(m)"],
    2: ["Sensor-2-1(m)", "Sensor-2-2(m)", "Sensor-2-3(m)", "Sensor-2-4(m)"],
    3: ["Sensor-3-1(m)", "Sensor-3-2(m)", "Sensor-3-3(m)", "Sensor-3-4(m)"],
    4: ["Sensor-4-1(m)", "Sensor-4-2(m)", "Sensor-4-3(m)", "Sensor-4-4(m)"],
}

gas_cols = ["MethConcStart(ppm)", "EthyConcStart(ppm)"]

# -----------------------------
def select_columns(df: pd.DataFrame, keep_cols: list) -> pd.DataFrame:
    """Return a DataFrame containing only the specified columns."""
    return df[keep_cols].copy()

# -----------------------------
# Start timer
start_time = time.time()

# -----------------------------
# Select file
selected_file = "4_slp_pos_em_denoised.csv"   # Change this to whichever file you want
df = pd.read_csv(selected_file)

# -----------------------------
# Choose sensor groups (example: groups 1 and 3)
selected_groups = [1, 3, 4]

# Collect columns from chosen groups
selected_sensor_cols = []
for g in selected_groups:
    selected_sensor_cols.extend(sensor_groups[g])

# Final feature set: selected sensors + gas
feature_cols = selected_sensor_cols 

# -----------------------------
# Select test data
x = select_columns(df, feature_cols)
x.to_csv(f"X_test_{selected_file}_{selected_groups}", index=False, float_format="%.6f")
# -----------------------------
# Predict anomalies on test set
y_pred = model.predict(x)

# Count anomalies and normals
n_anomalies = np.sum(y_pred == -1)
n_normals   = np.sum(y_pred == 1)

# Percentages
pct_anomalies = n_anomalies / len(y_pred) * 100
pct_normals   = n_normals / len(y_pred) * 100

# -----------------------------
# End timer
end_time = time.time()

# -----------------------------
# Print results
print("Execution time:", end_time - start_time, "seconds")
print("Test shape:", x.shape)
print(f"Files saved: X_test_{selected_file}_{selected_groups}")
print(f"Anomalies: {n_anomalies} ({pct_anomalies:.2f}%)")
print(f"Normals:   {n_normals} ({pct_normals:.2f}%)")

In [ ]:
import pandas as pd
import plotly.subplots as sp
import plotly.graph_objects as go

# -----------------------------
# Load dataset
selected_file = "2_em_denoised.csv"  # Change this to the desired category file
em_sig = pd.read_csv(selected_file, header=0)

# Convert all columns to numeric (non-numeric → NaN)
#em_sig = em_sig.apply(pd.to_numeric, errors="coerce").dropna()

# -----------------------------
# Define sensor groups (total 16 sensors)
sensor_groups = {
    "Sensor-1": ["Sensor-1-1(m)", "Sensor-1-2(m)", "Sensor-1-3(m)", "Sensor-1-4(m)"],
    "Sensor-2": ["Sensor-2-1(m)", "Sensor-2-2(m)", "Sensor-2-3(m)", "Sensor-2-4(m)"],
    "Sensor-3": ["Sensor-3-1(m)", "Sensor-3-2(m)", "Sensor-3-3(m)", "Sensor-3-4(m)"],
    "Sensor-4": ["Sensor-4-1(m)", "Sensor-4-2(m)", "Sensor-4-3(m)", "Sensor-4-4(m)"],
}

time_col = [c for c in em_sig.columns if "TimeStart" in c][0]  # auto-detect time column

# -----------------------------
# Collect all sensors into one list (16 sensors total)
all_sensors = []
for group in sensor_groups.values():
    all_sensors.extend(group)

# -----------------------------
# Create 16 vertically stacked subplots (one per sensor)
fig = sp.make_subplots(rows=len(all_sensors), cols=1,
                       subplot_titles=all_sensors,
                       shared_xaxes=True)

# Add slope plots for each sensor
for i, sensor in enumerate(all_sensors, start=1):
    fig.add_trace(
        go.Scatter(x=em_sig[time_col], y=em_sig[sensor],
                   mode="lines", name=sensor),
        row=i, col=1
    )
    fig.update_yaxes(title_text="Slope (m)", row=i, col=1)

# -----------------------------
# Update layout
fig.update_layout(
    height=3000, width=1000,  # large height to fit 16 subplots
    title_text=f"Time vs Slopes for All Sensors from {selected_file}",
    showlegend=False
)

# Common x-axis label
fig.update_xaxes(title_text="Time (s)", row=len(all_sensors), col=1)

fig.show()


In [ ]:
import pandas as pd
import plotly.subplots as sp
import plotly.graph_objects as go

# -----------------------------
# Load dataset
selected_file = "2_em_denoised.csv"  # Change this to the desired category file
em_sig = pd.read_csv(selected_file, header=0)

# Convert all columns to numeric (non-numeric → NaN)
em_sig = em_sig.apply(pd.to_numeric, errors="coerce").dropna()

# -----------------------------
# Detect time column automatically
time_col = [c for c in em_sig.columns if "TimeStart" in c][0]

# Define gas concentration columns
gas_cols = ["MethConcStart(ppm)", "EthyConcStart(ppm)"]

# -----------------------------
# Create subplots: one for each gas concentration
fig = sp.make_subplots(rows=len(gas_cols), cols=1,
                       subplot_titles=gas_cols,
                       shared_xaxes=True)

# Add concentration vs time plots
for i, gas in enumerate(gas_cols, start=1):
    fig.add_trace(
        go.Scatter(x=em_sig[time_col], y=em_sig[gas],
                   mode="lines", name=gas),
        row=i, col=1
    )
    fig.update_yaxes(title_text="Concentration (ppm)", row=i, col=1)

# -----------------------------
# Update layout
fig.update_layout(
    height=800, width=1000,
    title_text=f"Gas Concentrations vs Time from {selected_file}",
    showlegend=True
)

# Common x-axis label
fig.update_xaxes(title_text="Time (s)", row=len(gas_cols), col=1)

fig.show()


In [ ]:
import pandas as pd
import plotly.subplots as sp
import plotly.graph_objects as go

# -----------------------------
# Load dataset
selected_file = "2_em_denoised.csv"  # Change this to the desired category file
em_sig = pd.read_csv(selected_file, header=0)

# Convert all columns to numeric (non-numeric → NaN)
em_sig = em_sig.apply(pd.to_numeric, errors="coerce").dropna()

# -----------------------------
# Define sensor groups
sensor_groups = {
    "Sensor-1": ["Sensor-1-1(m)", "Sensor-1-2(m)", "Sensor-1-3(m)", "Sensor-1-4(m)"],
    "Sensor-2": ["Sensor-2-1(m)", "Sensor-2-2(m)", "Sensor-2-3(m)", "Sensor-2-4(m)"],
    "Sensor-3": ["Sensor-3-1(m)", "Sensor-3-2(m)", "Sensor-3-3(m)", "Sensor-3-4(m)"],
    "Sensor-4": ["Sensor-4-1(m)", "Sensor-4-2(m)", "Sensor-4-3(m)", "Sensor-4-4(m)"],
}

# Detect time column automatically
time_col = [c for c in em_sig.columns if "TimeStart" in c][0]

# Define gas concentration columns
gas_cols = ["MethConcStart(ppm)", "EthyConcStart(ppm)"]

# 👉 Select which group and sensor to plot
selected_group = "Sensor-1"   # change to "Sensor-2", "Sensor-3", or "Sensor-4"
selected_sensor = sensor_groups[selected_group][0]  # pick one sensor from the group

# -----------------------------
# Create two subplots:
# 1 = Time vs Gas Concentrations (both gases together)
# 2 = Time vs slope of one sensor
fig = sp.make_subplots(rows=2, cols=1,
                       subplot_titles=("Time vs Gas Concentrations",
                                       f"Time vs Slope ({selected_sensor})"),
                       shared_xaxes=True)

# --- Subplot 1: Time vs Gas Concentrations (overlayed lines)
fig.add_trace(
    go.Scatter(x=em_sig[time_col], y=em_sig[gas_cols[0]],
               mode="lines", name="Methane", line=dict(color="orange")),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=em_sig[time_col], y=em_sig[gas_cols[1]],
               mode="lines", name="Ethylene", line=dict(color="green")),
    row=1, col=1
)

# --- Subplot 2: Time vs slope of selected sensor
fig.add_trace(
    go.Scatter(x=em_sig[time_col], y=em_sig[selected_sensor],
               mode="lines", name=selected_sensor, line=dict(color="steelblue")),
    row=2, col=1
)

# -----------------------------
# Update layout
fig.update_layout(
    height=800, width=1000,
    title_text=f"Gas Concentrations and Sensor Slope from {selected_file}",
    showlegend=True
)

# Axis labels
fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_yaxes(title_text="Concentration (ppm)", row=1, col=1)
fig.update_yaxes(title_text="Slope (m)", row=2, col=1)

fig.show()


In [ ]:
import pandas as pd
import plotly.subplots as sp
import plotly.graph_objects as go

# Load dataset
selected_file = "2_em_denoised.csv"  # Change this to the desired category file
em_sig = pd.read_csv(selected_file, header=0)

# Convert all columns to numeric (non-numeric → NaN)
em_sig = em_sig.apply(pd.to_numeric, errors="coerce").dropna()

# Define sensor groups
sensor_groups = {
    "Sensor-1": ["Sensor-1-1(m)", "Sensor-1-2(m)", "Sensor-1-3(m)", "Sensor-1-4(m)"],
    "Sensor-2": ["Sensor-2-1(m)", "Sensor-2-2(m)", "Sensor-2-3(m)", "Sensor-2-4(m)"],
    "Sensor-3": ["Sensor-3-1(m)", "Sensor-3-2(m)", "Sensor-3-3(m)", "Sensor-3-4(m)"],
    "Sensor-4": ["Sensor-4-1(m)", "Sensor-4-2(m)", "Sensor-4-3(m)", "Sensor-4-4(m)"],
}

# 👉 Select which group to plot
selected_group = "Sensor-1"   # change to "Sensor-2", "Sensor-3", or "Sensor-4"

# Create vertically stacked subplots (one per sensor in the group)
fig = sp.make_subplots(rows=len(sensor_groups[selected_group]), cols=1,
                       subplot_titles=sensor_groups[selected_group])

# Add histograms with percentage labels
for i, sensor in enumerate(sensor_groups[selected_group], start=1):
    slopes = em_sig[sensor].dropna()

    # Histogram normalized to percent
    fig.add_trace(
        go.Histogram(
            x=slopes,
            nbinsx=30,
            name=sensor,
            marker_color="steelblue",
            histnorm="percent",   # normalize counts to percentages
            texttemplate="%{y:.2f}%",  # show percentage labels
            textposition="outside"
        ),
        row=i, col=1
    )

# Update layout
fig.update_layout(
    height=1000, width=900,
    title_text=f"Histogram of {selected_group} - from - {selected_file}",
    showlegend=False,
    bargap=0.1
)

# Axis labels
for i in range(1, len(sensor_groups[selected_group])+1):
    fig.update_xaxes(title_text="Slope Value", row=i, col=1)
    fig.update_yaxes(title_text="Percentage (%)", row=i, col=1)

fig.show()


In [ ]:
import pandas as pd
import plotly.subplots as sp
import plotly.graph_objects as go

# Load dataset
selected_file = "2_em_denoised.csv"  # Change this to the desired category file
em_sig = pd.read_csv(selected_file, header=0)

# Convert all columns to numeric (non-numeric → NaN)
em_sig = em_sig.apply(pd.to_numeric, errors="coerce").dropna()

# Define sensor groups
sensor_groups = {
    "Sensor-1": ["Sensor-1-1(m)", "Sensor-1-2(m)", "Sensor-1-3(m)", "Sensor-1-4(m)"],
    "Sensor-2": ["Sensor-2-1(m)", "Sensor-2-2(m)", "Sensor-2-3(m)", "Sensor-2-4(m)"],
    "Sensor-3": ["Sensor-3-1(m)", "Sensor-3-2(m)", "Sensor-3-3(m)", "Sensor-3-4(m)"],
    "Sensor-4": ["Sensor-4-1(m)", "Sensor-4-2(m)", "Sensor-4-3(m)", "Sensor-4-4(m)"],
}

# Loop through all groups and plot stacked histograms
for group_name, sensors in sensor_groups.items():
    # Create vertically stacked subplots (one per sensor in the group)
    fig = sp.make_subplots(rows=len(sensors), cols=1,
                           subplot_titles=sensors)

    # Add histograms for each sensor in the group
    for i, sensor in enumerate(sensors, start=1):
        slopes = em_sig[sensor].dropna()
        fig.add_trace(
            go.Histogram(x=slopes, nbinsx=30, name=sensor, marker_color="steelblue"),
            row=i, col=1
        )

    # Update layout
    fig.update_layout(
        height=1000, width=900,
        title_text=f"Histogram of {group_name} Group from file = {selected_file}",
        showlegend=False,
        bargap=0.1
    )

    # Axis labels
    for i in range(1, len(sensors)+1):
        fig.update_xaxes(title_text="Slope Value", row=i, col=1)
        fig.update_yaxes(title_text="Frequency", row=i, col=1)

    # Show figure for this group
    fig.show()
